# Train Forecasting Model

## Imports

In [25]:
import os
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error

import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

## Loading Data

In [26]:
PROCESSED_DATA_DIR= '../data/processed'
MODELS_DIR= '../models'

In [27]:
data_path= os.path.join(PROCESSED_DATA_DIR, 'forecasting_training_data.csv')

In [28]:
df= pd.read_csv(filepath_or_buffer= data_path)

In [29]:
df.head()

,sale_date,category,units_sold,daily_revenue
0,2016-09-15,health_beauty,3,134.97
1,2016-10-03,fashion_shoes,1,29.99
2,2016-10-03,furniture_decor,2,194.80
3,2016-10-03,sports_leisure,2,58.39
4,2016-10-03,toys,1,128.90


In [30]:
df.describe()

,units_sold,daily_revenue
count,18311.000000,18311.000000
mean,5.932936,712.442297
std,7.285815,969.197817
min,1.000000,3.850000
25%,1.000000,119.730000
50%,3.000000,349.900000
75%,8.000000,923.335000
max,192.000000,17667.020000


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sale_date      18311 non-null  object 
 1   category       18311 non-null  object 
 2   units_sold     18311 non-null  int64  
 3   daily_revenue  18311 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 572.3+ KB


In [32]:
df.shape

(18311, 4)

## Feature Engineering

In [33]:
# Converting sale_date to DateTime:
df['sale_date'] = pd.to_datetime(df['sale_date'])

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18311 entries, 0 to 18310
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   sale_date      18311 non-null  datetime64[ns]
 1   category       18311 non-null  object        
 2   units_sold     18311 non-null  int64         
 3   daily_revenue  18311 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 572.3+ KB


In [35]:
# Sorting DataFrame by category and sale_date:
df= df.sort_values(['category', 'sale_date']).reset_index(drop= True)

In [37]:
df.head(10)

,sale_date,category,units_sold,daily_revenue
0,2017-01-23,agro_industry_and_commerce,2,43.98
1,2017-01-31,agro_industry_and_commerce,1,21.99
2,2017-02-05,agro_industry_and_commerce,1,21.99
3,2017-02-08,agro_industry_and_commerce,1,21.99
4,2017-02-12,agro_industry_and_commerce,1,92.90
5,2017-02-13,agro_industry_and_commerce,1,21.99
6,2017-02-16,agro_industry_and_commerce,1,21.99
7,2017-02-18,agro_industry_and_commerce,1,21.99
8,2017-02-21,agro_industry_and_commerce,1,21.99
9,2017-03-17,agro_industry_and_commerce,1,59.99


*As there days missing for each category, we will add those days wth 0 units_sold and daily_revenue*

In [38]:
# Creating a Complete Date Range from Minimum to Maximum Date:
min_date, max_date= df['sale_date'].min(), df['sale_date'].max()

In [39]:
print(min_date, max_date)

2016-09-15 00:00:00 2018-08-29 00:00:00


In [42]:
full_date_range= pd.date_range(start= min_date,
                               end= max_date,
                               freq= 'D')

In [40]:
# Setting a Multi-index of Category + Sale_Date:
df= df.set_index(['category', 'sale_date'])

In [41]:
df.head()

units_sold  daily_revenue
category                   sale_date                            
agro_industry_and_commerce 2017-01-23           2          43.98
                           2017-01-31           1          21.99
                           2017-02-05           1          21.99
                           2017-02-08           1          21.99
                           2017-02-12           1          92.90

In [43]:
# Creating a New Index with All Combinations of Categories and Full Date Range:
new_index= pd.MultiIndex.from_product(
    [df.index.levels[0], full_date_range],
    names=['category', 'sale_date']
)

In [45]:
# Reindexing The Dataframe and filling newly created rows with 0:
df= df.reindex(index= new_index,
               fill_value= 0).reset_index()

In [46]:
df.head()

,category,sale_date,units_sold,daily_revenue
0,agro_industry_and_commerce,2016-09-15,0,0.0
1,agro_industry_and_commerce,2016-09-16,0,0.0
2,agro_industry_and_commerce,2016-09-17,0,0.0
3,agro_industry_and_commerce,2016-09-18,0,0.0
4,agro_industry_and_commerce,2016-09-19,0,0.0


In [47]:
# Extracting Time-Series Calendar Features:
df['year']= df['sale_date'].dt.year
df['month']= df['sale_date'].dt.month
df['day_of_week']= df['sale_date'].dt.dayofweek
df['day_of_year']= df['sale_date'].dt.dayofyear

In [48]:
df.head()

,category,sale_date,units_sold,daily_revenue,year,month,day_of_week,day_of_year
0,agro_industry_and_commerce,2016-09-15,0,0.0,2016,9,3,259
1,agro_industry_and_commerce,2016-09-16,0,0.0,2016,9,4,260
2,agro_industry_and_commerce,2016-09-17,0,0.0,2016,9,5,261
3,agro_industry_and_commerce,2016-09-18,0,0.0,2016,9,6,262
4,agro_industry_and_commerce,2016-09-19,0,0.0,2016,9,0,263


In [49]:
# Adding Auto-Regressive / LAG features:

# 1. Yesterday:
df['lag_1_revenue']= df.groupby('category')['daily_revenue'].shift(1)

# 2. Same Day, Last Week:
df['lag_7_revenue']= df.groupby('category')['daily_revenue'].shift(7)

# 3. EWMA of Last Week:
df['ewma_7_revenue']= df.groupby('category')['daily_revenue'].transform(
    lambda x: x.ewm(span= 7, adjust= False).mean().shift(1)
)

In [50]:
df.head()

,category,sale_date,units_sold,daily_revenue,year,month,day_of_week,day_of_year,lag_1_revenue,lag_7_revenue,ewma_7_revenue
0,agro_industry_and_commerce,2016-09-15,0,0.0,2016,9,3,259,NaN,NaN,NaN
1,agro_industry_and_commerce,2016-09-16,0,0.0,2016,9,4,260,0.0,NaN,0.0
2,agro_industry_and_commerce,2016-09-17,0,0.0,2016,9,5,261,0.0,NaN,0.0
3,agro_industry_and_commerce,2016-09-18,0,0.0,2016,9,6,262,0.0,NaN,0.0
4,agro_industry_and_commerce,2016-09-19,0,0.0,2016,9,0,263,0.0,NaN,0.0


In [51]:
# Dropping Null Values Created by LG Features:
df= df.dropna().reset_index(drop= True)

In [52]:
df.head()

,category,sale_date,units_sold,daily_revenue,year,month,day_of_week,day_of_year,lag_1_revenue,lag_7_revenue,ewma_7_revenue
0,agro_industry_and_commerce,2016-09-22,0,0.0,2016,9,3,266,0.0,0.0,0.0
1,agro_industry_and_commerce,2016-09-23,0,0.0,2016,9,4,267,0.0,0.0,0.0
2,agro_industry_and_commerce,2016-09-24,0,0.0,2016,9,5,268,0.0,0.0,0.0
3,agro_industry_and_commerce,2016-09-25,0,0.0,2016,9,6,269,0.0,0.0,0.0
4,agro_industry_and_commerce,2016-09-26,0,0.0,2016,9,0,270,0.0,0.0,0.0


## Train Test Split

In [53]:
# Cutoff Date for Last 30 Days:
cutoff_date= df['sale_date'].max() - pd.Timedelta(days= 30)

In [54]:
cutoff_date

Timestamp('2018-07-30 00:00:00')

In [55]:
# Splitting The Data:
train_df= df[df['sale_date'] < cutoff_date]
test_df= df[df['sale_date'] >= cutoff_date]

In [56]:
train_df.shape, test_df.shape

((47996, 11), (2201, 11))

In [57]:
# Defining Features and Target:
features= [
    'category', 'year', 'month', 'day_of_week', 'day_of_year',
    'lag_1_revenue', 'lag_7_revenue', 'ewma_7_revenue'
]
target= 'daily_revenue'

In [58]:
X_train, y_train= train_df[features], train_df[target]
X_test, y_test= test_df[features], test_df[target]

In [59]:
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(47996, 8) (47996,) (2201, 8) (2201,)


In [60]:
print(f"Training Range: {train_df['sale_date'].min().date()} to {train_df['sale_date'].max().date()}")

print(f"Testing Range: {test_df['sale_date'].min().date()} to {test_df['sale_date'].max().date()}")

Training Range: 2016-09-22 to 2018-07-29
Testing Range: 2018-07-30 to 2018-08-29


## Data Pre-Processing Pipeline

In [61]:
# Numeric and Categorical Features:
numeric_features= ['year', 'month', 'day_of_week', 'day_of_year', 'lag_1_revenue', 'lag_7_revenue', 'ewma_7_revenue']
categorical_features= ['category']

In [69]:
# Pre-Processing Pipeline for Numeric Features:
numeric_transformer= Pipeline(
    steps=[
        ('scaler', StandardScaler()),
    ]
)

# Pre-Processing Pipeline for Categoric Features:
categorical_transformer= Pipeline(
    steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [70]:
# Pre-Processing Pipeline:
preprocessor= ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## Model Training and Evaluation

In [71]:
# Initializing XGBoost Regressor:
xgb_regressor= xgb.XGBRegressor(
    n_estimators= 300,
    learning_rate= 0.05,
    max_depth= 6,
    subsample= 0.8,
    random_state= 42,
    n_jobs= -1
)

In [72]:
# Final Pipeline:
forecast_pipeline= Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', xgb_regressor)
    ]
)

In [73]:
# Training:
forecast_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['year', 'month',
                                                   'day_of_week', 'day_of_year',
                                                   'lag_1_revenue',
                                                   'lag_7_revenue',
                                                   'ewma_7_revenue']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['category'])])),
                ('regressor',
                 XGBRegressor(base_score=None, boos...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=300, n_jobs=-1,
                              num_parallel_tree=None, random_state=42, ...))])